In [1]:
import pandas as pd
import pickle
import numpy as np 
import random
from tqdm import tqdm
from pathlib import Path

# Create datasets for TIMIT MTurk task

## What this will create:

He will create a copy of every forground as 
For each set: 
* Create 170 trials with the following conditions: 
    * Distractor conditions 
      * 1-distractor
      * 2-distractor
      * 4-distractor
      * Speech-shaped noise
    * Signal to noise ratios:
      * -9 dB (for 1-distractor) 
      * -6 dB
      * -3 dB
      * 0 dB
      * +3 dB
    * Limit target words to not occur more than twice
    * Space out repeated target words across trials to minimize priming effects 
    * Balance distractor sex pairings 
        * eg 50-50 for 1-distractor; 25-25-25-25 for 4-distractor
    
All sets generated with random seed == 0 



### What's included in each final dataframe

Output 10 dataframes each with the following data.


##### Note: rows are trials and columns are trial stimuli and metadata.
original columns included: 

       'word', 'source', 'speaker', 'sr',
       'signal_length', 'speaker_sex', 'sentence_type', 'sentence_id',
       'dialect_region', 'data_split', 'signal', 'word_int'
      
with addition of:

        '_original_timit_index', '_original_cue_timit_index', 'cue_signal', 'cue_word', 'distractor', 
        '_original_distractor_timit_index', 'distractor_words', 'distractor_speakers', 'distractor_condition' 
        
where `_original_<type>_ix` maps audio sources to their original row in  `/om2/user/imgriff/datasets/timit/all_timit_wsn_compatible.pdpkl`
 
 
## Need to include catch trials in dataframes 

Existing files were not named, so it's easier to draw from timit that match wavs to unique talkers

## Load timit dataset

In [2]:
# Import timit data - this df contains all the relevant signals, labels and data 
path = '/om2/user/imgriff/datasets/timit/all_timit_wsn_compatible.pdpkl'

timit_excerpts = pd.read_pickle(path)


In [3]:
timit_excerpts['sentence_id'] = timit_excerpts['source'].apply(lambda x: x.split('-')[-1])

In [4]:
timit_excerpts.shape

(10309, 13)

In [5]:
## Made in timit_screen.ipynb on local desktop 

bad_sentences = ['sx74', 'sx144', 'sx376', 'si615', 'si670', 'si688', 'si693',
       'si917', 'si926', 'si927', 'si930', 'si931', 'si1078', 'si1102',
       'si1106', 'si1241', 'si1616', 'si1739', 'si1752', 'si1758',
       'si1767', 'si1933', 'si2020', 'si2040', 'si2064', 'si2112',
       'si2134', 'si2204', 'si2284']


timit_excerpts = timit_excerpts[~timit_excerpts.sentence_id.isin(bad_sentences)]

In [6]:
timit_excerpts.shape

(10239, 13)

In [7]:
## Set random seed for exp generation (init to 0) 

np.random.seed(0)
random.seed(0)

In [8]:
# Check that we can get sex balance 

In [9]:
timit_excerpts.speaker_sex.value_counts()

m    7075
f    3164
Name: speaker_sex, dtype: int64

In [10]:
timit_excerpts[timit_excerpts.word_int != 0].speaker_sex.value_counts()

m    604
f    277
Name: speaker_sex, dtype: int64

### Normalize timit audio before cutting

In [11]:
all_signals = np.stack(timit_excerpts.signal)

In [12]:
all_signals = all_signals - all_signals.mean(1).reshape(-1,1) # de-mean rows 


In [13]:
per_signal_rms = np.sqrt(np.mean(np.power(all_signals, 2), axis=1)).reshape(-1,1) # per signal rms 


In [14]:
all_signals = all_signals * 0.1 / per_signal_rms # normalize all signals to rms of 0.02 or ~60dB

In [15]:
timit_excerpts['signal'] = [signal for signal in all_signals] # update signal column

## Segment into target and cue portions of TIMIT 

Targets and distractors will be drawn from target_timit, while cues will come from cue_timit. This way, cues will never contain target words, but distractors will. 

Distractors will be drawn from the complimentary set of targets for a given experiment, so no excerpt will be both a target and distractor in a given experiment. 

In [16]:
timit_excerpts.sentence_id.value_counts() # see most frequent sentences 

sa1       1479
sa2       1373
sx174       24
sx371       23
sx378       22
          ... 
si561        1
si1366       1
si736        1
si895        1
si563        1
Name: sentence_id, Length: 1952, dtype: int64

In [17]:
# parse target & rest 

target_timit = timit_excerpts[timit_excerpts.word_int != 0]
# target_sentences = target_timit.sentence_id.unique()

# filter cues 
cue_timit = timit_excerpts[timit_excerpts.word_int == 0]
# cue_timit = timit_excerpts[(timit_excerpts.word_int == 0) & (~timit_excerpts.sentence_id.str.contains('1|2'))]

# cut targets to include talkers included in both cue and target lest 
# # Cut targets to minimize word repetition 
valid_words = target_timit.word.value_counts()[target_timit.word.value_counts() < 5].index

# get training stim for start of experiment 
training_timit = target_timit[~target_timit.word.isin(valid_words)] # can use words with more reps for training set

target_timit = target_timit[target_timit.word.isin(valid_words)]

valid_talkers = np.intersect1d(target_timit.speaker.unique(),cue_timit.speaker.unique(), assume_unique=True)
target_timit = target_timit[target_timit.speaker.isin(valid_talkers)]



In [18]:
target_timit.source

23        train-dr1-fdaw0-sx326
53         train-dr1-fecd0-sx68
61       train-dr1-fetb0-si1778
63        train-dr1-fetb0-si518
70         train-dr1-fetb0-sx68
                  ...          
10221      test-dr8-mjln0-sx369
10230     test-dr8-mjtc0-si1460
10231     test-dr8-mjtc0-si2090
10275      test-dr8-mpam0-sx199
10284     test-dr8-mres0-si1217
Name: source, Length: 406, dtype: object

In [19]:
cue_timit.groupby('speaker').apply(lambda group: group[group.source!='train-dr1-fecd0-sx68'].sample(2))

word  _full_dataset_index                  source speaker  \
speaker                                                                        
fadg0   8549   sprained                 5255    test-dr4-fadg0-sx109   fadg0   
        8540     greasy                 5250      test-dr4-fadg0-sa1   fadg0   
faem0   608      greasy                  380     train-dr2-faem0-sa1   faem0   
        621         pie                  388   train-dr2-faem0-sx402   faem0   
fajw0   628        time                  393  train-dr2-fajw0-si1893   fajw0   
...                 ...                  ...                     ...     ...   
mwsh0   5329          a                 3272  train-dr5-mwsh0-si1426   mwsh0   
mwvw0   8093        are                 4983   test-dr2-mwvw0-si2106   mwvw0   
        8100       will                 4987    test-dr2-mwvw0-sx306   mwvw0   
mzmb0   1872  testimony                 1132  train-dr2-mzmb0-si1166   mzmb0   
        1867       suit                 1130     train-dr2-mzmb0-sa1   mzmb0   

                 sr  signal_length speaker_sex sentence_type sentence_id  \
speaker                                                                    
fadg0   8549  20000          40000           f            sx       sx109   
        8540  20000          40000           f            sa         sa1   
faem0   608   20000          40000           f            sa         sa1   
        621   20000          40000           f            sx       sx402   
fajw0   628   20000          40000           f            si      si1893   
...             ...            ...         ...           ...         ...   
mwsh0   5329  20000          40000           m            si      si1426   
mwvw0   8093  20000          40000           m            si      si2106   
        8100  20000          40000           m            sx       sx306   
mzmb0   1872  20000          40000           m            si      si1166   
        1867  20000          40000           m            sa         sa1   

             dialect_region data_split  \
speaker                                  
fadg0   8549            dr4       test   
        8540            dr4       test   
faem0   608             dr2      train   
        621             dr2      train   
fajw0   628             dr2      train   
...                     ...        ...   
mwsh0   5329            dr5      train   
mwvw0   8093            dr2       test   
        8100            dr2       test   
mzmb0   1872            dr2      train   
        1867            dr2      train   

                                                         signal  word_int  
speaker                                                                    
fadg0   8549  [-0.0010429479785946621, -0.000343901111583348...         0  
        8540  [-0.0004952403478393673, -0.000687535392179104...         0  
faem0   608   [-0.0012968726510258447, -0.001265264027094789...         0  
        621   [-0.006784420582333832, 0.018174181256428102, ...         0  
fajw0   628   [-0.001361284677081394, -0.0012524564090383402...         0  
...                                                         ...       ...  
mwsh0   5329  [0.10320394974793663, 0.1527861883756019, 0.16...         0  
mwvw0   8093  [0.006764491584082795, 0.0016704211281750344, ...         0  
        8100  [0.0011639277585173166, 0.0005686747737540318,...         0  
mzmb0   1872  [-0.1140901993410487, -0.10225367066892674, -0...         0  
        1867  [0.0786436030512453, 0.1486626473421042, -0.06...         0  

[1260 rows x 13 columns]

In [20]:
target_timit.shape

(406, 13)

In [21]:
# Save df of catch trials & demo cue selection logic

# pull 10 training examples 

train_targets = target_timit.groupby('speaker_sex').sample(n=10).copy() # replace=False is default
train_targets.drop(columns=['_full_dataset_index', 'data_split'], inplace=True)

train_targets.index = train_targets.index.set_names(['_original_timit_index']) # add original index mapping
train_targets = train_targets.reset_index() # Reset for 0:10 ixs 

# get set of distractors given targets
bg_ixs = target_timit[["speaker", 'sentence_id','word']].merge(train_targets[['speaker', 'sentence_id','word']],
                                how = 'outer' ,indicator=True).loc[lambda x : x['_merge']=='left_only'].index
bg_timit = target_timit.iloc[bg_ixs]
bg_timit.index = bg_timit.index.set_names(['_original_timit_index']) # keep original ixs 
bg_timit = bg_timit.reset_index()

print(f"{len(train_targets.word.unique())} unique words ")
print(f"{train_targets.word.value_counts().max()} max frequency ")

# Sample cues 
## To sample cues, get list of talkers in target set 

talkers = train_targets.speaker.unique()

# sample 1 cue per excerpt from viable cues 
samples_per_talker = {talker:count for talker,count in train_targets.speaker.value_counts().items()}
viables_cues = cue_timit[cue_timit.speaker.isin(talkers)]

cues = viables_cues.groupby('speaker').apply(lambda group: group.sample(samples_per_talker[group.name]))
cues.drop(columns='speaker', inplace=True)
cues = cues.reset_index()
cues.rename(columns={'level_1':'_original_cue_timit_index'}, inplace=True)

train_targets.sort_values(by='speaker', inplace=True)
train_targets.reset_index(inplace=True, drop=True)


cues.sort_values(by='speaker', inplace=True)
cues.reset_index(inplace=True, drop=True)



### Merge cues with foregrounds  
cues[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker']] = cues[['signal', 'word', '_original_cue_timit_index', 'speaker']]
# Combine as experiment dataframe
training_df = train_targets.join(cues[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker']])
assert (training_df['speaker'] == training_df['cue_speaker']).all(), "Cue and Target talkers don't match!"




20 unique words 
1 max frequency 


In [22]:
training_df.head()

,_original_timit_index,word,source,speaker,sr,signal_length,speaker_sex,sentence_type,sentence_id,dialect_region,signal,word_int,cue_signal,cue_word,_original_cue_timit_index,cue_speaker
0,23,novel,train-dr1-fdaw0-sx326,fdaw0,20000,40000,f,sx,sx326,dr1,"[0.011782246730868976, 0.08233421110820222, 0....",461,"[-0.08059113233432856, -0.1574949309033704, -0...",an,14,fdaw0
1,7554,become,test-dr1-felc0-sx306,felc0,20000,40000,f,sx,sx306,dr1,"[-0.149552760494615, -0.18440199070296268, -0....",76,"[-0.07331340547644653, -0.07308784018363555, -...",greasy,7543,felc0
2,4376,difficult,train-dr5-fgdp0-sx448,fgdp0,20000,40000,f,sx,sx448,dr5,"[0.00139015216211521, 0.0007251573614166049, 0...",203,"[-0.007427113597063375, -0.014920978401971072,...",heroic,4367,fgdp0
3,9180,house,test-dr5-fhes0-sx389,fhes0,20000,40000,f,sx,sx389,dr5,"[0.0002804323043918055, 0.0006113662873129717,...",321,"[0.16474580322706292, 0.21244699506143175, 0.2...",a,9167,fhes0
4,7566,quite,test-dr1-fjem0-si634,fjem0,20000,40000,f,si,si634,dr1,"[0.03028287467191142, 0.036875799966606865, 0....",566,"[0.01904497844170738, 0.027698903427361518, 0....",more,7573,fjem0


In [23]:
out_name = Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/attn_task_training_excerpts_0_1rms_mturk.pdpkl')

training_df.to_pickle(out_name)

In [24]:
 # Now that we're using 170 trials, we can cut down to 85 unique foregrounds if need be 
valid_words = target_timit.word.value_counts()[target_timit.word.value_counts() < 5].index
target_timit[target_timit.word.isin(valid_words)].speaker_sex.value_counts()

m    285
f    121
Name: speaker_sex, dtype: int64

In [25]:
target_timit.shape

(406, 13)

In [26]:
timit_excerpts.columns

Index(['word', '_full_dataset_index', 'source', 'speaker', 'sr',
       'signal_length', 'speaker_sex', 'sentence_type', 'sentence_id',
       'dialect_region', 'data_split', 'signal', 'word_int'],
      dtype='object')

##  Make Dataframes

###  Define functions:



Define some useful functions for mixing audio

In [27]:
next_pow_2 = lambda x: int(pow(2, np.ceil(np.log2(x))))

def combine_with_noise(clean, noise, snr):
    # get ratio in rms 
    rms_ratio = np.power(10.0, snr / 20.0)
    
    # remove DC of each signal
    clean = clean - clean.mean()
    noise = noise - noise.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    noise_rms = np.sqrt(np.mean(np.power(noise, 2)))
    # scale factor for setting noise to desired SNR 
    scale_factor = clean_rms / (noise_rms * rms_ratio)
    # Blend signals 
    noise = noise * scale_factor
    mixture = clean + noise[:len(clean)]
    return mixture, scale_factor

def rms_normalize(wav, new_rms=0.1, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor

## Define windowing function - will apply a cosine ramp to the start and end of a signal

In [28]:
## Precompute spectrum of all signals for speech shaped noise 

all_foregrounds = np.stack(target_timit['signal'])
n_x = all_foregrounds.shape[-1]
n_fft = next_pow_2(n_x)
X = np.fft.rfft(all_foregrounds, next_pow_2(n_fft))
X = X.mean(0)

def generate_speech_shaped_noise(X=X):
    # Randomize phase.
    noise_mag = np.abs(X) * np.exp(
        2 * np.pi * 1j * np.random.random(X.shape[-1]))
    noise = np.real(np.fft.irfft(noise_mag, n_fft))
    ssn = noise[:n_x]
    return ssn

## Make datasets 

### Setup experimental conditions 


In [29]:
## Make trial conditions 
import itertools

n_trials = 170 


n_male = n_female = n_trials//2 # force balance of foregrounds 
n_datasets = 25

# setup 4x4 design
snrs = [-6, -3, 0, 3] # 10/20/2022 Changed from -15, -5, 0, 10
backgrounds = [1,2,4,'ssn']

condition_pairs = list(itertools.product(snrs, backgrounds))

condition_pairs.append((-9,1)) # only want -9dB for 1 distractor

n_catch_trials = 12 
n_conditions = len(condition_pairs)

trials_per_condition = n_trials // n_conditions

distractor_types = ['m', 'f']

two_talker_conds = list(itertools.combinations_with_replacement(distractor_types,2))
four_talker_conds = list(itertools.combinations_with_replacement(distractor_types,4))


file_out_dir = Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes_mturk/')

In [30]:
for set_ix in tqdm(range(n_datasets)):
    # Randomly generate experimental condition list
    all_conditions = []
    for condition_pair in condition_pairs:
        snr, condition = condition_pair
        for ix in range(trials_per_condition):
            if condition == 1:
                distractor = np.random.choice(distractor_types)
            elif condition == 2:
                type_ix = np.random.choice(len(two_talker_conds))
                distractor = two_talker_conds[type_ix]
            elif condition == 4:
                type_ix = np.random.choice(len(four_talker_conds))
                distractor = four_talker_conds[type_ix]
            else:
                distractor = 'ssn'

            all_conditions.append({'snr':snr,
                                  'condition':condition,
                                  'distractor':distractor})

    # shuffle condition order inplace
    random.shuffle(all_conditions)
    
    # sample our talkers for one set 
    set_targets = target_timit.groupby('speaker_sex').sample(n=n_female).copy() # replace=False is default
    set_targets.drop(columns=['_full_dataset_index', 'data_split'], inplace=True)

    set_targets.index = set_targets.index.set_names(['_original_timit_index']) # add original index mapping
    set_targets = set_targets.reset_index() # Reset for 0:399 ixs 
    
    # get set of distractors given targets
    bg_ixs = target_timit[["speaker", 'sentence_id','word']].merge(set_targets[['speaker', 'sentence_id','word']],
                                    how = 'outer' ,indicator=True).loc[lambda x : x['_merge']=='left_only'].index
    bg_timit = target_timit.iloc[bg_ixs]
    bg_timit.index = bg_timit.index.set_names(['_original_timit_index']) # keep original ixs 
    bg_timit = bg_timit.reset_index()
    
    print(f"{len(set_targets.word.unique())} unique words in dataset {set_ix}")
    print(f"{set_targets.word.value_counts().max()} max frequency in dataset {set_ix}")
    
    # Sample cues 
    ## To sample cues, get list of talkers in target set 

    talkers = set_targets.speaker.unique()

    # sample 1 cue per excerpt from viable cues 
    samples_per_talker = {talker:count for talker,count in set_targets.speaker.value_counts().items()}
    viables_cues = cue_timit[cue_timit.speaker.isin(talkers)]

    cues = viables_cues.groupby('speaker').apply(lambda group: group.sample(samples_per_talker[group.name]))
    cues.drop(columns='speaker', inplace=True)
    cues = cues.reset_index()
    cues.rename(columns={'level_1':'_original_cue_timit_index'}, inplace=True)
    
    set_targets.sort_values(by='speaker', inplace=True)
    set_targets.reset_index(inplace=True, drop=True)
    
    ### Merge cues with foregrounds  
    cues[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker', 'cue_source']] = cues[['signal', 'word', '_original_cue_timit_index', 'speaker', 'source']]
    # Combine as experiment dataframe
    exp_df = set_targets.join(cues[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker', 'cue_source']])
    assert (exp_df['speaker'] == exp_df['cue_speaker']).all(), "Cue and Target talkers don't match!"
    
    # Generate trial by trial stim    
    trial_data = {'mixture_signal':[],
            'distractor_signal':[],
            '_original_distractor_timit_indices':[],
            'distractor_words':[],
            'distractor_speakers':[],
            'distractor_conditions':[],
            'distractor_sex':[],
            'snrs':[]}
    cue_signals = []
    cue_snrs = []

    # Loop over trials to make mixtures 
    for i in range(n_trials):
        # get signal
        signal = exp_df['signal'][i]
        speaker = exp_df['speaker'][i]
        valid_distractors = bg_timit[bg_timit['speaker'] != speaker].reset_index(drop=True) # reset so ixing works
        # get condition 
        snr, condition, distractors = all_conditions[i].values()
        # Get data based on condition
        if condition == 'ssn':
            noise = generate_speech_shaped_noise()
            indices = 'ssn'
            words = 'ssn'
            speakers = 'ssn'

        else:
            if condition == 1:
                distractor_ixs = np.where(valid_distractors.speaker_sex==distractors)[0]
                distractor_ixs = np.random.choice(distractor_ixs, size=1)
                noise = valid_distractors['signal'][distractor_ixs[0]]

            elif condition in [2, 4]:
                talker_sex_counts = dict(zip(*np.unique(distractors, return_counts = True)))
                m_distractors=[]
                f_distractors=[]
                if 'm' in talker_sex_counts:
                    m_distractors = np.where(valid_distractors.speaker_sex =='m')[0]
                    m_distractors = np.random.choice(m_distractors, size=talker_sex_counts['m'])
                if 'f' in talker_sex_counts:
                    f_distractors = np.where(valid_distractors.speaker_sex=='f')[0]
                    f_distractors = np.random.choice(f_distractors, size=talker_sex_counts['f'])
                distractor_ixs = np.concatenate([f_distractors, m_distractors])

                noise = np.stack(valid_distractors['signal'][distractor_ixs]).sum(0) # sum unique distractors 
            # get common feats 
            words = valid_distractors['word'][distractor_ixs].values
            indices = valid_distractors['_original_timit_index'][distractor_ixs].values
            speakers = valid_distractors['speaker'][distractor_ixs].values
        
        # create trial stim 
        noise,_ = rms_normalize(noise)
        signal, _ = rms_normalize(signal)
        mixture, _ = combine_with_noise(signal, noise, snr) # first_scale_factor unused
        mixture, mixture_scale_factor = rms_normalize(mixture)
        cue, _ = rms_normalize(exp_df['cue_signal'][i])
        # rove cue
        cue_signal = cue * mixture_scale_factor
       
        cue_snrs.append(snr)
        cue_signals.append(cue_signal)

        # save values for tiral 
        trial_data['mixture_signal'].append(mixture)
        trial_data['distractor_signal'].append(noise)
        trial_data['_original_distractor_timit_indices'].append(indices)
        trial_data['distractor_words'].append(words)
        trial_data['distractor_speakers'].append(speakers)
        trial_data['distractor_conditions'].append(condition)

        trial_data['distractor_sex'].append(''.join(str(token) for token in distractors))
        trial_data['snrs'].append(snr)
    # convert to trial df 
    trial_data = pd.DataFrame.from_dict(trial_data) 
    # merge with exp_df 
    exp_df = exp_df.join(trial_data)
    # update cues with roved level
    exp_df['cue_snr'] = cue_snrs
    exp_df['cue_signal'] = cue_signals
    
    
    # make sure cue is not same excerpt as target. If it is, get a new cue
    cue_target_match = exp_df[exp_df.source == exp_df.cue_source]
    if len(cue_target_match) > 0: 
        print(f"{len(cue_target_match)} cues to re draw")
        rows_to_fix = cue_target_match.index
        for row in rows_to_fix:
            target_voice, source = exp_df.loc[row, ['speaker', 'source']]
            new_cue = viables_cues[(viables_cues.speaker==target_voice) & (viables_cues.source != source)].sample(1)
            # Update causes warning, but is currently safe 
            exp_df.loc[row, ['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker', 'cue_source']] = [v for v in new_cue[['signal', 'word', '_full_dataset_index', 'speaker', 'source']].iloc[0]]

        
    
    # get catch trials
    catch_trials = bg_timit.sample(n=n_catch_trials)
    catch_trials.drop(columns=['_full_dataset_index', 'data_split'], inplace=True)

    catch_trials.sort_values(by='speaker', inplace=True)
    catch_trials.reset_index(inplace=True, drop=True)
    
    #  catch cues are just the target signal
    catch_trials[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker', 'cue_source']] = catch_trials[['signal', 'word', '_original_timit_index', 'speaker', 'source']]
    
    catch_trials['mixture_signal'] = catch_trials['signal']
    catch_trials['distractor_signal'] = 'catch trial'
    catch_trials['_original_distractor_timit_indices'] = 'catch trial'
    catch_trials['distractor_words'] = 'catch trial'
    catch_trials['distractor_speakers']  = 'catch trial'
    catch_trials['distractor_conditions'] = 'catch trial'
    catch_trials['snrs'] = 'catch trial'
    
    exp_df = pd.concat([exp_df, catch_trials],ignore_index=True)
    if not file_out_dir.exists():
        file_out_dir.mkdir(parents=True, exist_ok=True)
#     break
    ## Save exp_df 
    df_name = file_out_dir / f"attn_task_dataset_{set_ix:02d}.pdpkl"
    exp_df.to_pickle(df_name)
    

  0%|          | 0/25 [00:00<?, ?it/s]

139 unique words in dataset 0
3 max frequency in dataset 0
8 cues to re draw


/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return asarray(a).ndim
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return asarray(a).ndim
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which

131 unique words in dataset 1
4 max frequency in dataset 1
6 cues to re draw


/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return asarray(a).ndim
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return asarray(a).ndim
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which

...

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return asarray(a).ndim
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return asarray(a).ndim
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which

140 unique words in dataset 23
3 max frequency in dataset 23
11 cues to re draw


/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return asarray(a).ndim
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return asarray(a).ndim
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which

138 unique words in dataset 24
3 max frequency in dataset 24
11 cues to re draw


/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return asarray(a).ndim
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return asarray(a).ndim
/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3199: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which

In [47]:
from IPython.display import Audio

# Audio(exp_df['cue_signal'][0], rate=20000, normalize=False)

# Demo Distractor by SNR conditions 

In [48]:
isi = np.zeros(int(0.25 * 20_000))

def make_trial(cue, mixture):
    return np.hstack([cue, isi, mixture])

### 1 distractor at -6, -3, 0, and +3 dB SNR

In [49]:
one_talker_df = exp_df[exp_df['distractor_conditions'] == 1]

In [50]:
trial = one_talker_df.loc[one_talker_df['snrs'] == -6].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print("word = ", trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

word =  lower


In [ ]:
trial = one_talker_df.loc[one_talker_df['snrs'] == -3].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = one_talker_df.loc[one_talker_df['snrs'] == 0].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [54]:
trial = one_talker_df.loc[one_talker_df['snrs'] == 3].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

their


### 2 distractor at -6, -3, 0, and +3 dB SNR

In [ ]:
two_talker_df = exp_df[exp_df['distractor_conditions'] == 2]

In [ ]:
trial = two_talker_df.loc[two_talker_df['snrs'] == -8].iloc[2]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = two_talker_df.loc[two_talker_df['snrs'] == -3].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = two_talker_df.loc[two_talker_df['snrs'] == 0].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = two_talker_df.loc[two_talker_df['snrs'] == 7].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

### 4 distractor at -6, -3, 0, and +3 dB SNR

In [ ]:
four_talker_df = exp_df[exp_df['distractor_conditions'] == 4]

In [ ]:
trial = four_talker_df.loc[four_talker_df['snrs'] == -8].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = four_talker_df.loc[four_talker_df['snrs'] == -3].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = four_talker_df.loc[four_talker_df['snrs'] == 0].iloc[1]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = four_talker_df.loc[four_talker_df['snrs'] == 7].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

### Speech Shaped Noise distractor at -6, -3, 0, and +3 dB SNR

In [ ]:
ssn_df = exp_df[exp_df['distractor_conditions'] == 'ssn']

In [ ]:
trial = ssn_df.loc[ssn_df['snrs'] == -8].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = ssn_df.loc[ssn_df['snrs'] == -3].iloc[20]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = ssn_df.loc[ssn_df['snrs'] == 0].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

In [ ]:
trial = ssn_df.loc[ssn_df['snrs'] == 7].iloc[0]
cue, mixture = trial[['cue_signal', 'mixture_signal']]
trial_signal = make_trial(cue, mixture)
print(trial['word'])
Audio(trial_signal, rate=20000, normalize=False)

## Demo Speech Shaped noise gen

In [ ]:
def make_speech_shaped_noise(x):
    ## Use for demos 
    x = np.asarray(x)
    n_x = x.shape[-1]
    n_fft = next_pow_2(n_x)
    X = np.fft.rfft(x, next_pow_2(n_fft))
    if X.ndim == 2:
        X = X.mean(0)
    # Randomize phase.
    noise_mag = np.abs(X) * np.exp(
        2 * np.pi * 1j * np.random.random(X.shape[-1]))
    noise = np.real(np.fft.irfft(noise_mag, n_fft))
    out = noise[:n_x]

    return out

ssn = make_speech_shaped_noise(np.stack(target_timit['signal']))

In [ ]:
ssn.shape

In [ ]:
import matplotlib.pyplot as plt
plt.plot(np.abs(np.fft.rfft(ssn)))

In [ ]:
plt.plot(np.abs(np.fft.rfft(exp_df['signal'][100])))

In [ ]:
from IPython.display import Audio
Audio(ssn, rate=20000)

In [ ]:
Audio(exp_df['signal'][10], rate=20000)

In [ ]:
Audio(exp_df['mixture_signal'][4], rate=20000)